# Point-to-Point OFDM Systems

Consider 4 cases: 
1. SISO
2. SIMO
3. MISO
4. MIMO

In [ ]:
import numpy as np

def GenerateNoise(signal: np.ndarray, SNR: int) -> tuple:
    SNRlinear = 10 ** (SNR / 10)
    SignalPower = np.mean(np.power(np.abs(signal), 2), axis=0)
    NoisePower = SignalPower / SNRlinear
    C_Noise = np.diag(np.sqrt(NoisePower.reshape(-1)))
    Noise = np.sqrt(1 / 2) * C_Noise @ (np.random.randn(*signal.shape) + 1j * np.random.randn(*signal.shape))
    return Noise, C_Noise

def ComputeNSME(EstChan:np.ndarray, GroundChan: np.ndarray) -> float:
    numerator = np.mean(np.power(np.abs(EstChan - GroundChan), 2))
    denumerator = np.mean(np.power(np.abs(GroundChan), 2))
    NMSEdB = 10 * np.log10((numerator / denumerator))
    return float(NMSEdB)

def GenRandomDiag(N, K, dtype="complex", repeat=1):
    if dtype == "int":
        d = np.random.randint(1,10,(N,K))
        A = np.zeros((N,K*repeat,K*repeat), dtype=int)
    else: 
        d = (np.random.randn(N,K) + np.random.randn(N,K)*1j) / np.sqrt(2)
        A = np.zeros((N,K*repeat,K*repeat), dtype=complex)
    for i in range(repeat):
        A[:, np.arange(i*K,(i+1)*K), np.arange(i*K,(i+1)*K)] = d
    return A

def disp(param, num_tab, val):
    text = param + ":" + num_tab * "\t" +"{}"
    print(text.format(val))

class LSEstimator():
    def __init__(self, Pilot: np.ndarray) -> None:
        """
        Solving N equations of y = Ax + w,
        where - y, w have shape of (m, 1)
              - A has shape of (m, n)
              - x has shape of (n, 1)

        Parameters
        --------------------
        Pilot (A)   : np.ndarray of shape (N, m, n)        

        Note: Shapes of Pilots and ActualChan depend on the system.
        """
        self.Pilot = Pilot

    def estimate(self, Rx):
        EstimatedChannel = np.zeros_like(Rx, dtype=complex)
        for i in range(Rx.shape[0]):
            EstimatedChannel[i] = Rx[i] / np.diag(self.Pilot[i]).reshape(-1,1)
        return EstimatedChannel
    
class LMMSEEstimator():
    def __init__(self, Pilot: np.ndarray, ActualChan: np.ndarray, NoisePower: np.ndarray) -> None:
        """
        Solving N equations of y = Ax + w,
        where - y, w have shape of (m, 1)
              - A has shape of (m, n)
              - x has shape of (n, 1)

        Closed-form Solutions
        --------------------
        x_hat = mu_x + R_xx  * (R_xy)^-1 * (y - mu_y), where R is correlation matrix
        x_hat = mu_x + C_xx * A^H (A * C_xx * A^H + var_noise * I)^-1 * (y - A * mu_x), where C is covariance matrix

        Parameters
        --------------------
        Pilot (A)       : np.ndarray of shape (N, m, n)
        ActualChan (x)  : np.ndarray of shape (N, n, 1)
        NoisePower      : np.ndarray of shape (m, m)

        Note: Shapes of Pilots and ActualChan depend on the system.
        """
        # Compute first- and second-order stat of channel in the frequency domain.
        self.mean_h = np.mean(ActualChan, axis=0)
        self.R_hh = np.mean(ActualChan @ np.transpose(ActualChan,(0,2,1)).conj(), axis=0)
        self.NoisePower = np.diag(NoisePower)
        self.Pilot = Pilot 
        
    def estimate(self, Rx) -> np.ndarray:

        EstimatedChannel = np.zeros_like(Rx, dtype=complex)
        for i in range(Rx.shape[0]):
            X = self.Pilot[i]
            EstimatedChannel[i] = self.mean_h + self.R_hh @ X.conj().T @ np.linalg.inv(X @ self.R_hh @ X.conj().T + self.NoisePower) @ (Rx[i] - X @ self.mean_h)
        return EstimatedChannel



# SISO OFDM System

Transmitter sends pilot $K$ symbols, $\bold{x} \in \mathrm{C}^{K}$ to the receiver. The received symbols $\bold{y} \in \mathrm{C}^{K}$ can be modeled as $\bold{y} = \bold{H} \bold{x} + \bold{w}$, where $\bold{H}$ is a diagonal matrix resulted from the OFDM system. As $\bold{H}$ is diagonal matrix and $\bold{x}$ is a vector, their elements can be exchanged.

Thus, $\bold{y} = \bold{X} \bold{h} + \bold{w}$

In [20]:
# SISO setting
Nr = 1
Nt = 1
K = 10
N = 1000
SNR = 10

X = GenRandomDiag(N,K, "int")   # frequency pilot symbols
h = (np.random.randn(N,K,1) + 1j*np.random.randn(N,K,1)) / np.sqrt(2)  # channel in freq-domain
ChannelOutput = X @ h
w, VarNoise = GenerateNoise(ChannelOutput, SNR)
y = ChannelOutput + w

LS_Estimator = LSEstimator(X)
LMMSE_Estimator = LMMSEEstimator(X, h, VarNoise)

LSEstimatedChannel = LS_Estimator.estimate(y)
LMMSEEstimatedChannel = LMMSE_Estimator.estimate(y)

LS_NMSE = ComputeNSME(LSEstimatedChannel, h)
LMMSE_NMSE = ComputeNSME(LMMSEEstimatedChannel, h)

print("\n--- SUMMARY ---")
disp("LS NSME [dB]",2,round(LS_NMSE, 4))
disp("LMMSE NSME [dB]",1,round(LMMSE_NMSE, 4))


--- SUMMARY ---
LS NSME [dB]:		-2.672
LMMSE NSME [dB]:	-3.7331


# SIMO OFDM System

Single antenna transmitter sends pilot of $K$ symbols, $\bold{x} \in \mathrm{C}^{K}$ to $N_r$-antenna receiver. The received symbols at $i$-th antenna $\bold{y}_i \in \mathrm{C}^{K}$ can be modeled as $\bold{y}_i = \bold{H}_i \bold{x} + \bold{w}$, where $\bold{H}_i$ is a diagonal channel matrix between first pair of transceiver resulted from the OFDM system. Agian, as $\bold{H}_i$ is diagonal matrix and $\bold{x}$ is a vector, their elements can be exchanged.

Thus, $\bold{y}_i = \bold{X} \bold{h}_i + \bold{w}$. Stacking the received symbols of all $N_r$ antennas, resulting in

$\tilde{\bold{y}} = \tilde{\bold{X}} \tilde{\bold{h}}$, where $\tilde{\bold{y}} = [\bold{y}_1^T ... \bold{y}_{N_r}^T]^T \in \mathrm{C}^{KN_r}$,
$\tilde{\bold{X}} = BlkDiag(\bold{X} ... \bold{X}) \in \mathrm{C}^{K N_r \times K N_r}$, and $\tilde{\bold{h}} = [\bold{h}_1^T ... \bold{h}_{N_r}^T]^T \in \mathrm{C}^{KN_r}$

In [22]:
# SIMO setting
Nr = 6
Nt = 1
K = 10
N = 1000
SNR = 10

X = GenRandomDiag(N, K, "int", repeat=Nr)   # frequency pilot symbols
h = (np.random.randn(N,K*Nr,1) + np.random.randn(N,K*Nr,1) * 1j) / np.sqrt(2)  # channel in freq-domain
ChannelOutput = X @ h
w, VarNoise = GenerateNoise(ChannelOutput, SNR)
y = ChannelOutput + w

LMMSE_Estimator = LMMSEEstimator(X, h, VarNoise)
LS_Estimator = LSEstimator(X)

LSEstimatedChannel = LS_Estimator.estimate(y)
LMMSEEstimatedChannel = LMMSE_Estimator.estimate(y)

LS_NMSE = ComputeNSME(LSEstimatedChannel, h)
LMMSE_NMSE = ComputeNSME(LMMSEEstimatedChannel, h)

print("\n--- SUMMARY ---")
disp("LS NSME [dB]",2,round(LS_NMSE, 4))
disp("LMMSE NSME [dB]",1,round(LMMSE_NMSE, 4))


--- SUMMARY ---
LS NSME [dB]:		-2.6161
LMMSE NSME [dB]:	-2.7421


# MISO OFDM System

$N_t$-antenna transmitter sends pilot of $K$ symbols, $\bold{x} \in \mathrm{C}^{K}$ to a single-antenna receiver. The received symbols $\bold{y} \in \mathrm{C}^{K}$ can be modeled as $\bold{y} = \sum_i^{N_t} \bold{H}_i \bold{x}_i + \bold{w}_i$, where $\bold{H}_i$ is a diagonal channel matrix between first pair of transceiver resulted from the OFDM system. 

Thus, $\bold{y} = \sum_i^{N_t} \bold{X}_i \bold{h}_i + \bold{w}_i = \tilde{\bold{X}} \tilde{\bold{h}}$, where 
$\tilde{\bold{X}} =[\bold{X}_1 ... \bold{X}_{N_t}] \in \mathrm{C}^{K \times K N_t}$, and $\tilde{\bold{h}} = [\bold{h}_1^T ... \bold{h}_{N_t}^T]^T \in \mathrm{C}^{KN_t}$

For this MISO system, the matrix $\tilde{\bold{X}}$ is wide, therefore the pilots must be orthogonal.

In [ ]:
#### NOT DONE YET ####
# MISO setting
Nr = 1
Nt = 2
K = 10
N = 1000
SNR = 10

X = GenRandomDiag(N, K, "int", repeat=Nr)   # frequency pilot symbols
h = (np.random.randn(N,K*Nr,1) + np.random.randn(N,K*Nr,1) * 1j) / np.sqrt(2)  # channel in freq-domain
ChannelOutput = X @ h
w, VarNoise = GenerateNoise(ChannelOutput, SNR)
y = ChannelOutput + w

LMMSE_Estimator = LMMSEEstimator(X, h, VarNoise)
LS_Estimator = LSEstimator(X)

LSEstimatedChannel = LS_Estimator.estimate(y)
LMMSEEstimatedChannel = LMMSE_Estimator.estimate(y)

LS_NMSE = ComputeNSME(LSEstimatedChannel, h)
LMMSE_NMSE = ComputeNSME(LMMSEEstimatedChannel, h)

print("\n--- SUMMARY ---")
disp("LS NSME [dB]",2,round(LS_NMSE, 4))
disp("LMMSE NSME [dB]",1,round(LMMSE_NMSE, 4))


--- SUMMARY ---
LS NSME [dB]:		-2.6376
LMMSE NSME [dB]:	-3.7544
